In [4]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Already loaded — just clean it
loans = loans_raw.copy()

print(f"Raw shape: {loans.shape}")

# ── STEP 1: Create binary TARGET from loan_status ──────────────────
default_statuses = [
    'Charged Off',
    'Default',
    'Late (31-120 days)',
    'Does not meet the credit policy. Status:Charged Off'
]
paid_statuses = [
    'Fully Paid',
    'Does not meet the credit policy. Status:Fully Paid'
]
final_statuses = default_statuses + paid_statuses

# Keep only resolved loans — drop Current, In Grace Period, Late (16-30)
loans = loans[loans['loan_status'].isin(final_statuses)].copy()
loans['TARGET'] = loans['loan_status'].isin(default_statuses).astype('int')

print(f"After status filter: {len(loans):,} rows")
print(f"Default rate: {loans['TARGET'].mean()*100:.2f}%")
print(f"\nTARGET distribution:")
print(loans['TARGET'].value_counts())

# ── STEP 2: Drop data leakage columns ──────────────────────────────
leakage_cols = [
    'loan_status', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp',
    'total_rec_int', 'total_rec_late_fee', 'recoveries',
    'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt',
    'next_pymnt_d', 'last_credit_pull_d', 'out_prncp', 'out_prncp_inv',
    # Extra leakage cols in this extended dataset
    'debt_settlement_flag_date', 'settlement_status', 'settlement_date',
    'settlement_amount', 'settlement_percentage', 'settlement_term',
    'hardship_loan_status', 'hardship_payoff_balance_amount',
    'hardship_last_payment_amount', 'orig_projected_additional_accrued_interest'
]
before = loans.shape[1]
loans.drop(columns=[c for c in leakage_cols if c in loans.columns], inplace=True)
print(f"\nDropped {before - loans.shape[1]} leakage columns")

# ── STEP 3: Drop columns with > 40% nulls ──────────────────────────
null_pct = loans.isnull().sum() / len(loans)
high_null = null_pct[null_pct > 0.4].index
loans.drop(columns=high_null, inplace=True)
print(f"Dropped {len(high_null)} columns with >40% nulls")
print(f"Remaining columns: {loans.shape[1]}")

# ── STEP 4: Fix data types ──────────────────────────────────────────
# int_rate: "10.5%" → 10.5
loans['int_rate'] = loans['int_rate'].astype('str').str.replace('%','').astype('float')

# revol_util: "45.2%" → 45.2
loans['revol_util'] = loans['revol_util'].astype('str').str.replace('%','')
loans['revol_util'] = pd.to_numeric(loans['revol_util'], errors='coerce')

# emp_length: "10+ years" → 10, "< 1 year" → 0
loans['emp_length_num'] = loans['emp_length'].str.extract('(\d+)').astype('float')

# issue_d: "Jan-2015" → datetime
loans['issue_date']  = pd.to_datetime(loans['issue_d'], format='%b-%Y', errors='coerce')
loans['issue_year']  = loans['issue_date'].dt.year
loans['issue_month'] = loans['issue_date'].dt.month

# term: " 36 months" → 36
loans['term_months'] = loans['term'].str.extract('(\d+)').astype('float')

print("\nDtype fixes done")

# ── STEP 5: Impute remaining nulls ─────────────────────────────────
num_cols = loans.select_dtypes('number').columns
cat_cols = loans.select_dtypes('object').columns
loans[num_cols] = loans[num_cols].fillna(loans[num_cols].median())
loans[cat_cols] = loans[cat_cols].fillna('Unknown')
print(f"Nulls remaining: {loans.isnull().sum().sum()}")

# ── STEP 6: Feature engineering ────────────────────────────────────
loans['income_to_loan']     = loans['annual_inc'] / loans['loan_amnt'].clip(lower=1)
loans['dti_risk_flag']      = (loans['dti'] > 30).astype('int')
loans['fico_midpoint']      = (loans['fico_range_low'] + loans['fico_range_high']) / 2
loans['has_delinquency']    = (loans['delinq_2yrs'] > 0).astype('int')
loans['rate_grade_spread']  = loans['int_rate'] - loans.groupby('grade')['int_rate'].transform('mean')
loans['loan_to_income_pct'] = (loans['loan_amnt'] / loans['annual_inc'].clip(lower=1) * 100).round(2)
loans['is_high_grade']      = loans['grade'].isin(['A', 'B']).astype('int')

print("\nFeature engineering done")
print(f"\nFinal shape: {loans.shape}")
print(f"Default rate: {loans['TARGET'].mean()*100:.2f}%")
print(f"\nGrade vs default rate:")
print(loans.groupby('grade')['TARGET'].mean().round(3))

# ── STEP 7: Save ───────────────────────────────────────────────────
loans.to_parquet(r'E:\finsight\data\processed\loans_clean.parquet', index=False)
print("\n✓ Saved to data/processed/loans_clean.parquet")

Raw shape: (2260668, 145)
After status filter: 1,328,284 rows
Default rate: 21.41%

TARGET distribution:
TARGET
0    1043940
1     284344
Name: count, dtype: int64

Dropped 24 leakage columns
Dropped 47 columns with >40% nulls
Remaining columns: 75

Dtype fixes done
Nulls remaining: 0


KeyError: 'fico_range_low'

In [2]:
import pandas as pd

# Reload the raw file without any cleaning
loans_raw = pd.read_csv(r'E:\finsight\data\raw\loan.csv', low_memory=False, nrows=5)

print("Columns:")
print(loans_raw.columns.tolist())
print(f"\nFirst column name: '{loans_raw.columns[0]}'")
print(f"\nShape of first 5 rows: {loans_raw.shape}")
print(f"\nFirst row values:")
print(loans_raw.iloc[0])

Columns:
['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code', 'application_type', 'annual_inc_joint', 'dti_joint', 'verification_status_joint', 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'open_acc_6m', 'open_act_il', 'open_il_12m'

In [3]:
# Check the last few rows too
loans_raw = pd.read_csv(r'E:\finsight\data\raw\loan.csv', low_memory=False)

print(f"Total rows loaded: {len(loans_raw)}")
print(f"\nFirst 3 rows - loan_amnt and id:")
print(loans_raw[['id', 'loan_amnt', 'loan_status', 'grade']].head(3))
print(f"\nLast 3 rows:")
print(loans_raw[['id', 'loan_amnt', 'loan_status', 'grade']].tail(3))
print(f"\nloan_status value counts:")
print(loans_raw['loan_status'].value_counts())

Total rows loaded: 2260668

First 3 rows - loan_amnt and id:
   id  loan_amnt loan_status grade
0 NaN       2500     Current     C
1 NaN      30000     Current     D
2 NaN       5000     Current     D

Last 3 rows:
         id  loan_amnt loan_status grade
2260665 NaN      10000     Current     B
2260666 NaN      12000     Current     D
2260667 NaN      16550     Current     D

loan_status value counts:
loan_status
Fully Paid                                             1041952
Current                                                 919695
Charged Off                                             261655
Late (31-120 days)                                       21897
In Grace Period                                           8952
Late (16-30 days)                                         3737
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     31
Name: count, dt

In [5]:
# Check what FICO columns exist
fico_cols = [c for c in loans.columns if 'fico' in c.lower()]
print("FICO columns:", fico_cols)
print("\nAll current columns:")
print(sorted(loans.columns.tolist()))

FICO columns: []

All current columns:
['TARGET', 'acc_now_delinq', 'acc_open_past_24mths', 'addr_state', 'annual_inc', 'application_type', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util', 'chargeoff_within_12_mths', 'collections_12_mths_ex_med', 'debt_settlement_flag', 'delinq_2yrs', 'delinq_amnt', 'disbursement_method', 'dti', 'dti_risk_flag', 'earliest_cr_line', 'emp_length', 'emp_length_num', 'emp_title', 'funded_amnt', 'funded_amnt_inv', 'grade', 'hardship_flag', 'home_ownership', 'income_to_loan', 'initial_list_status', 'inq_last_6mths', 'installment', 'int_rate', 'issue_d', 'issue_date', 'issue_month', 'issue_year', 'loan_amnt', 'mo_sin_old_il_acct', 'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl', 'mort_acc', 'mths_since_recent_bc', 'mths_since_recent_inq', 'num_accts_ever_120_pd', 'num_actv_bc_tl', 'num_actv_rev_tl', 'num_bc_sats', 'num_bc_tl', 'num_il_tl', 'num_op_rev_tl', 'num_rev_accts', 'num_rev_tl_bal_gt_0', 'num_sats', 'num_tl_120dpd_2m', 'num_tl_30dpd', 'nu

In [6]:
# ── STEP 6: Feature engineering (FICO-free version) ────────────────
loans['income_to_loan']     = loans['annual_inc'] / loans['loan_amnt'].clip(lower=1)
loans['dti_risk_flag']      = (loans['dti'] > 30).astype('int')
loans['has_delinquency']    = (loans['delinq_2yrs'] > 0).astype('int')
loans['rate_grade_spread']  = loans['int_rate'] - loans.groupby('grade')['int_rate'].transform('mean')
loans['loan_to_income_pct'] = (loans['loan_amnt'] / loans['annual_inc'].clip(lower=1) * 100).round(2)
loans['is_high_grade']      = loans['grade'].isin(['A', 'B']).astype('int')
loans['has_pub_rec']        = (loans['pub_rec'] > 0).astype('int')
loans['high_revol_util']    = (loans['revol_util'] > 80).astype('int')

print("Feature engineering done")
print(f"\nFinal shape: {loans.shape}")
print(f"Default rate: {loans['TARGET'].mean()*100:.2f}%")
print(f"\nGrade vs default rate:")
print(loans.groupby('grade')['TARGET'].mean().round(3))

# ── STEP 7: Save ───────────────────────────────────────────────────
loans.to_parquet(r'E:\finsight\data\processed\loans_clean.parquet', index=False)
print("\n✓ Saved to data/processed/loans_clean.parquet")

Feature engineering done

Final shape: (1328284, 88)
Default rate: 21.41%

Grade vs default rate:
grade
A    0.066
B    0.145
C    0.241
D    0.322
E    0.400
F    0.465
G    0.512
Name: TARGET, dtype: float64

✓ Saved to data/processed/loans_clean.parquet
